## Program 1A

The most basic implementation of the outlined DCD theory. Uses three matrices to perform an O(N^2) scaling evaluation. The first two matrices perform an all-by-all evaluation. The third matrix provides a mask to disqualify any future waypoints from the current C2 pose. Complete with lap timer for specific stages.

In [55]:
n_rows = 1

In [56]:
import pandas as pd
import numpy as np
import time
from time import perf_counter_ns

h_1 = 20
r_1 = 20
h_2 = 10
r_2 = 20

file_path = r"C:\Users\Smith\OneDrive\MSc Project\05 Final Code\Bunny Head Raw Lines for Computation Testing.xlsx"

df = pd.read_excel(
    file_path,
    sheet_name=0,
    usecols="A:F",
    nrows=n_rows,
    header=None,
    engine="openpyxl"
)

df.columns = ["x", "y", "z", "i", "j", "k"]
A = df.to_numpy(dtype=float)

In [57]:
def point_in_CTC(A, h_1, r_1, h_2, r_2, eps=1e-12):

    t_start = perf_counter_ns()
    t_last = t_start

    def lap(name):
        nonlocal t_last
        now = perf_counter_ns()
        dt_ms = (now - t_last) / 1_000_000
        total_ms = (now - t_start) / 1_000_000
        print(f"{name:<45} /{dt_ms:10.3f}/ ms   total: {total_ms:10.3f} ms")
        t_last = now

    A = np.asarray(A, dtype=float)
    lap("Input to numpy array")

    P = A[:, 0:3]   # xyz points
    U = A[:, 3:6]   # unit orientation vectors
    N = A.shape[0]
    lap("Separate Points and Vectors")

    # Create "t": N x N t value matrix for all tip/point combinations
    point_dp_axis = U @ P.T
    tip_dp_axis = np.einsum("ij,ij->i", P, U)
    t = point_dp_axis - tip_dp_axis[:, None]
    lap("Create t matrix")

    # Create "d_perp_sq": N x N shortest distance value matrix
    p2 = np.einsum("ij,ij->i", P, P)
    v2 = p2[:, None] + p2[None, :] - 2.0 * (P @ P.T)
    v2 = np.maximum(v2, 0.0)
    d_perp_sq = v2 - t*t
    d_perp_sq = np.maximum(d_perp_sq, 0.0)
    lap("Create d_perp_sq matrix")

    # Mask for where j < i
    idx = np.arange(N)
    previous_mask = idx[None, :] < idx[:, None]
    lap("Create mask")

    # Axial slab test
    axial_ok = (t >= -eps) & (t <= h_1 + h_2 + eps)
    lap("Slab Test")

    # Shortest distance test
    cone_region = t <= h_1 + eps
    cylinder_region = t > h_1 + eps

    cone_ok = (h_1*h_1*d_perp_sq) <= (r_1*r_1*t*t + eps)
    cylinder_ok = d_perp_sq <= (r_2*r_2 + eps)

    radial_ok = (cone_region & cone_ok) | (cylinder_region & cylinder_ok)
    lap("Radial distance test")

    # Create intersection matrix
    inside = previous_mask & axial_ok & radial_ok
    lap("Matrix Combination")

    # Get compact list of hit pairs [current_row_i, previous_point_j]
    hit_pairs = np.argwhere(inside)
    lap("Hit List")

    print("-" * 75)
    print(f"{'TOTAL':<45} {(perf_counter_ns() - t_start) / 1_000_000:10.3f} ms")

    return hit_pairs

In [58]:
start = time.perf_counter()
hit_pairs = point_in_CTC(A, h_1, r_1, h_2, r_2)
elapsed_ms = (time.perf_counter() - start) * 1000
print(f"Function time: {elapsed_ms:.3f} ms")
print(hit_pairs.shape)

Input to numpy array                          /     0.009/ ms   total:      0.009 ms
Separate Points and Vectors                   /     0.297/ ms   total:      0.306 ms
Create t matrix                               /    44.228/ ms   total:     44.535 ms
Create d_perp_sq matrix                       /     2.139/ ms   total:     46.674 ms
Create mask                                   /     0.537/ ms   total:     47.211 ms
Slab Test                                     /     0.880/ ms   total:     48.091 ms
Radial distance test                          /     0.750/ ms   total:     48.841 ms
Matrix Combination                            /     0.090/ ms   total:     48.932 ms
Hit List                                      /     0.327/ ms   total:     49.259 ms
---------------------------------------------------------------------------
TOTAL                                             49.319 ms
Function time: 49.723 ms
(0, 2)
